# Generate cleaned Discharge Summary Samples using DischargeNoteSummarization Task

This notebook demonstrates the usage of MIMIC-IV Note dataset and DischargeNoteSummarizationTask to generate discharge summary samples for LLM training.

In [ ]:
import os
from pyhealth.datasets import MIMIC4Dataset
from pyhealth.tasks import BaseTask
from pyhealth.data import Patient
from typing import List, Dict, Any
from pyhealth.processors import TextProcessor
import argparse
import random
import pandas as pd
from pathlib import Path

pd.options.mode.chained_assignment = None
import re
import pickle
import nltk
from collections import Counter
from tqdm import tqdm
import string

# Initialize the MIMI4Dataset using the note data downloaded from Physionet website.

Name of dataset used is discharge.csv.gz from Physionet : https://physionet.org/content/ann-pt-summ/1.0.1/

In [ ]:
full_note_dataset = MIMIC4Dataset(
    note_root='/content/drive/llm_data/',
    note_tables=["discharge"]
)

Memory usage Starting MIMIC4Dataset init: 882.8 MB


INFO:pyhealth.datasets.mimic4:Memory usage Starting MIMIC4Dataset init: 882.8 MB


Initializing mimic4 dataset from None|/content/drive/MyDrive/llm_data/|None (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing mimic4 dataset from None|/content/drive/MyDrive/llm_data/|None (dev mode: False)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232


...

INFO:pyhealth.datasets.mimic4:Memory usage After initializing mimic4_note: 883.0 MB


Memory usage After Note dataset initialization: 883.0 MB


INFO:pyhealth.datasets.mimic4:Memory usage After Note dataset initialization: 883.0 MB


Memory usage Completed MIMIC4Dataset init: 883.0 MB


INFO:pyhealth.datasets.mimic4:Memory usage Completed MIMIC4Dataset init: 883.0 MB


In [ ]:
full_note_dataset.stats()

# Print an event using a patient id

In [ ]:
print(full_note_dataset.get_patient('10000032').get_events())

# Define the DischargeNoteSummarization Task

Create DischargeNoteSummarization class , initialize the input and output schema.
Extract specific sections "Brief Hospital Course" and "Discharge Instructions". Clean the samples to remove extra spaces and new lines to create a paragraph for each sample text.



In [ ]:

from typing import Dict, List, Any, Tuple, Union

class DischargeNoteSummarization(BaseTask):
    task_name: str = "DischargeNoteSummarization"

    input_schema: Dict[str , str] = {
      "subject_id" : "text",
      "hadm_id": "text",
      "text": "text"
    }

    output_schema: Dict[str, str] = {
        "brief_hospital_course": "text",
        "summary": "text"
    }


    def __call__(self, patient: Patient) -> List[Dict[str, Any]]:
      samples = []
      subject_id = patient.patient_id
      for dis in patient.get_events("discharge"):

          textNote = dis.attr_dict['text']
          hadm_id = dis.attr_dict['hadm_id']

          ## Extract the brief_hospital_course

          start = textNote.find("Brief Hospital Course:")
          if start < 0:
              #brief_hospital_course = None
              continue
          end = textNote.find("Medications on Admission:")
          if end == -1:
              end = textNote.find("Discharge Medications:")
          if end == -1:
              end = textNote.find("Discharge Disposition:")
          if end == 0 or start >= end:
              continue
          brief_hospital_course = textNote[start: end].replace('\n',  ' ')
          brief_hospital_course = ' '.join(brief_hospital_course.split())
          # Quality check
          num_words = len(textNote.split(' '))
          
          #extract the summary
          start = textNote.find("Discharge Instructions:")
          end = textNote.find("Followup Instructions:")
          if start < 0 or end < 0:
              continue
          summary = textNote[start: end].replace('\n',  ' ')
          summary = ' '.join(summary.split())
          if len(summary) == 0 or len(summary) < 350:
            continue
          summary = summary.strip()



          samples.append({
            "text":textNote,
            "brief_hospital_course": brief_hospital_course,
            "summary" : summary,
            "subject_id" : subject_id,
            "hadm_id": hadm_id
          })

      return samples

In [ ]:
! rm -r /root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld

# Run the Discharge Note Summarization Task

Run the DischargeNoteSummarization Task with 4 workers and note dataset

In [ ]:
task = DischargeNoteSummarization()
processed_dataset = full_note_dataset.set_task(task ,num_workers=4)

Setting task PatientNoteProcessingTask for mimic4 base dataset...


INFO:pyhealth.datasets.base_dataset:Setting task PatientNoteProcessingTask for mimic4 base dataset...


Task cache paths: task_df=/root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/task_df.ld, samples=/root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Task cache paths: task_df=/root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/task_df.ld, samples=/root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


Applying task transformations on data with 4 workers...


...

  1%|▏         | 3584/257238 [00:00<00:22, 11325.59it/s]/usr/local/lib/python3.12/dist-packages/litdata/streaming/writer.py:284: UserWarning: An item was larger than the target chunk size (64.0 MB). The current chunk will be 64.0 MB in size.
  warnings.warn(
100%|██████████| 257238/257238 [00:51<00:00, 5013.98it/s]

Worker 0 finished processing samples.



INFO:pyhealth.datasets.base_dataset:Worker 0 finished processing samples.


Cached processed samples to /root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


INFO:pyhealth.datasets.base_dataset:Cached processed samples to /root/.cache/pyhealth/98de9a11-0af5-5cd9-81f2-2da31802c232/tasks/PatientNoteProcessingTask_46bb372d-34eb-5a38-bd99-ca6f30f0f026/samples_cdbbc602-34e2-5a41-8643-4c76b08829f6.ld


In [ ]:

mimic_df = pd.DataFrame(processed_dataset)


<class 'pyhealth.datasets.sample_dataset.SampleDataset'>


# Print the dataframe head

In [ ]:
print(mimic_df.head())

In [ ]:
len(mimic_df.iloc[1]['summary'])

740

# Perform further processing on the dataframe

Run more data cleaning tasks on the mimic_df

In [ ]:
import swifter
re_service = re.compile(r'^Service: (.*)$', re.IGNORECASE|re.MULTILINE)  # Either after Serive:
re_service_extra = re.compile(r'^Date of Birth:.*Sex:\s{0,10}\w\s{0,10}___: (.*)$', re.IGNORECASE|re.MULTILINE)  # Fallback if deidentified

mimic_df['service'] = mimic_df['text'].swifter.apply(lambda s: re_service.search(s).group(1) if re_service.search(s) is not None else None)


In [ ]:
mimic_df.loc[~mimic_df['service'].notnull(), 'service'] = mimic_df.loc[~mimic_df['service'].notnull(), 'text'].swifter.apply(lambda s: re_service_extra.search(s).group(1) if re_service_extra.search(s) is not None else None)


Pandas Apply:   0%|          | 0/183 [00:00<?, ?it/s]

In [ ]:
mimic_df.loc[~mimic_df['service'].notnull(), 'service'] = ''
mimic_df['service'] = mimic_df['service'].str.strip()
mimic_df['service'] = mimic_df['service'].str.strip(string.punctuation)

In [ ]:
mimic_df.head()

In [ ]:
mimic_df.loc[:, 'summary'] = mimic_df['summary'].str.strip()
print("  Remove all leading and trailing whitespaces in each line of text")
mimic_df.loc[:, 'summary'] = mimic_df['summary'].apply(lambda s: '\n'.join([x.strip() for x in s.split('\n')]))

  Remove all leading and trailing whitespaces in each line of text


In [ ]:
def change_why_what_next_pattern_to_text(summaries):
    """ Change all occurrences of the static why, what, next pattern occuring in many MIMIC summaries to fluent text. """

    # Determine random string used instead of headings
    random_string = ''.join(random.choices(string.ascii_uppercase + string.digits, k=20)) + '\n- '  # Replace removed dash
    #summaries = summaries.apply(lambda s: WHY_WHAT_NEXT_HEADINGS_DASHED_LIST.sub(random_string, s))
    # Now replace all items after the random string, end of paragraph marked by double newline
    dash_regex = re.compile(r'(?:\.)?\n-\s{0,4}', re.MULTILINE|re.IGNORECASE)  # Also removes
    # Filter summaries that contain random_string, replace dash with fullstop and whitespace
    def remove_dash_from_paragraphs(s):
        paragraphs = s.split(random_string)
        # For each paragraph remove all dashes until \n\n
        res = [paragraphs[0]]
        paragraphs = paragraphs[1:]
        for p in paragraphs:
            items = p.split('\n\n', 1)[0]
            items = '. '.join(dash_regex.split(items))
            if '\n\n' in p:
                items = items + '\n\n' + p.split('\n\n', 1)[1]
            res.append(items.strip())
        return '\n\n'.join(res)
    summaries = summaries.apply(lambda s: remove_dash_from_paragraphs(s) if random_string in s else s)
    return summaries

In [ ]:
print(f"  Change Why in hospital / What was done / What next pattern into fluent text.")
mimic_df.loc[:, 'summary'] = change_why_what_next_pattern_to_text(mimic_df['summary'])

  Change Why in hospital / What was done / What next pattern into fluent text.


In [ ]:
import regular_expressions

# Remove all subheadings that are not followed by a list
subheading_regex = re.compile(regular_expressions.WHY_WHAT_NEXT_HEADINGS, re.MULTILINE|re.IGNORECASE)
print(f"  Remove {mimic_df['summary'].apply(lambda s: subheading_regex.findall(s)).apply(len).sum()} list template headings.")
mimic_df.loc[:, 'summary'] = mimic_df['summary'].apply(lambda s: subheading_regex.sub('\n', s))

  Remove 0 list template headings.


In [ ]:
print("  Remove all newlines in continuous text")
mimic_df.loc[:, 'summary'] = mimic_df['summary'].apply(lambda s: regular_expressions.re_newline_in_text.sub(' ', s))
mimic_df.loc[:, 'summary'] = mimic_df['summary'].apply(lambda s: regular_expressions.re_multiple_whitespace.sub(' ', s))

  Remove all newlines in continuous text


In [ ]:
for replacement, regex in regular_expressions.SIMPLE_DEIDENTIFICATION_PATTERNS:
  # Print number of replacements
  print(f"    Replace {mimic_df['summary'].apply(lambda s: len(regex.findall(s))).sum()} ___ with \'{replacement}\'.")
  mimic_df.loc[:, 'summary'] = mimic_df['summary'].apply(lambda s: re.sub(regex, replacement, s))


    Replace 5306 ___ with 'You '.
    Replace 308 ___ with ' you'.
    Replace 327 ___ with ' your '.
    Replace 0 ___ with ' your '.


In [ ]:
old_len = len(mimic_df)
max_double_newlines = 5
mimic_df = mimic_df[mimic_df['summary'].map(lambda s: s.count('\n\n')) <= max_double_newlines]
print(f"  Removed {old_len - len(mimic_df)} summaries with more than {max_double_newlines} double newlines.")


  Removed 0 summaries with more than 5 double newlines.


In [ ]:
import nltk
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [ ]:
mimic_df['sentences'] = mimic_df['summary'].swifter.apply(lambda s: list(nltk.sent_tokenize(s)))
mimic_df = mimic_df[mimic_df['sentences'].map(len) >= 3]
print(f"  Removed {old_len - len(mimic_df)} summaries with less than 3 sentences.")


In [ ]:
print(f"  Combine all sentences with single whitespaces.")
mimic_df.loc[:, 'summary'] = mimic_df['sentences'].swifter.apply(lambda s: regular_expressions.re_whitespace.sub(' ', ' '.join(s)))
mimic_df.drop(columns=['sentences'], inplace=True)

  Combine all sentences with single whitespaces.


Pandas Apply:   0%|          | 0/251248 [00:00<?, ?it/s]

In [ ]:
print(f"  Count occurrences of ___ in each summary.")
mimic_df['num_deidentified'] = mimic_df['summary'].swifter.apply(lambda s: s.count('___'))

  Count occurrences of ___ in each summary.


Pandas Apply:   0%|          | 0/251248 [00:00<?, ?it/s]

In [ ]:
mimic_df.head()

In [ ]:
num_words_per_deidentified = 10
old_len = len(mimic_df)
mimic_df = mimic_df[mimic_df['num_deidentified'] <= mimic_df['summary'].map(lambda s: len(s.split(' ')) / num_words_per_deidentified)]
# mimic_df = mimic_df[mimic_df['num_deidentified'] <= 10]
print(f"  Removed {old_len - len(mimic_df)} summaries with more than one ___ per {num_words_per_deidentified} words.")


  Removed 965 summaries with more than one ___ per 10 words.


In [ ]:

old_len = len(mimic_df)
mimic_df.drop_duplicates(subset=['subject_id'], keep='first', inplace=True)
print(f"  Removed {old_len - len(mimic_df)} / {old_len} notes from same hospital stay.")
old_len = len(mimic_df)
re_ds = re.compile(r"Discharge Instructions:\n", re.IGNORECASE)
mimic_df = mimic_df[mimic_df['text'].str.contains(re_ds, regex=True)]

  Removed 0 / 122083 notes from same hospital stay.


# Generate processed summaries and save to drive

Save the processed summaries to drive in csv format. These cleaned summaries will be used as training data for Large Language Models to generate high quality patient summaries and to identify hallucinations.

In [ ]:
outputpath = f"{nb_path}/outputdata/"

print(Path(outputpath))

print(f"\nOutput data to {outputpath}")
# Create output directory if it does not exist
Path(f"{nb_path}/outputdata").mkdir(parents=True, exist_ok=True)
mimic_df.to_csv('/content/drive/outputdata/mimic_processed_summaries.csv', index=False)

In [ ]:
print(len(mimic_df))

122083
